In [1]:
# 1) Imports and configuration
import os
import time
import random
import numpy as np
import torch
from torch.utils.data import IterableDataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

data_path = "/home/pk222/projects/PDEControl_DPC/datasets/heat_smooth_f_dataset.npz"
samples_to_load = 100
seed = 32
train_frac = 0.8
batch_size = 75
num_steps = 1000
log_iter = 10
lr = 1e-3
transition_steps = 2000
decay_rate = 0.9
sample_id = 5
num_samples_to_evaluate = 10

resume_path = ""
resume_strict = True
save_best = True
save_last = True


In [2]:
# 2) Reproducibility and device
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required but not available.")
device = torch.device("cuda")
print(f"Using device: {device}")


Using device: cuda


In [3]:
# 3) Data loading
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found: {data_path}")

dataset = np.load(data_path)
solutions = torch.from_numpy(dataset["solutions"][:samples_to_load])
controls = torch.from_numpy(dataset["controls"][:samples_to_load])
x_cord = torch.from_numpy(dataset["x"]).reshape(-1, 1)
dt = dataset["dt"]

print(f"Loaded {samples_to_load}/{dataset['solutions'].shape[0]} samples:")
print(f"solutions: {tuple(solutions.shape)}")
print(f"controls: {tuple(controls.shape)}")
print(f"x: {tuple(x_cord.shape)}")
print(f"dt: {dt}")

u_t = solutions[:, :-1, :]
u_next = solutions[:, 1:, :]
f_t = controls[:, :, :]

print(f"u_t: {tuple(u_t.shape)}")
print(f"u_next: {tuple(u_next.shape)}")
print(f"f_t: {tuple(f_t.shape)}")


Loaded 100/3000 samples:
solutions: (100, 401, 100)
controls: (100, 400, 4)
x: (100, 1)
dt: 0.001
u_t: (100, 400, 100)
u_next: (100, 400, 100)
f_t: (100, 400, 4)
